# Task A Run 11 -- one reinitialized layer, and the blend weight it deserves

The current best Task A submission, `0.8187`, is a fixed blend of a full-fit TF-IDF/SVM and
a full-data MuRIL. Its MuRIL component reinitializes the top **two** encoder layers, a
default inherited rather than chosen, and the 57/43 blend weights were fitted in Run 8
**against that two-layer component**.

Change the component and the weight it deserves changes with it. A stronger MuRIL should
take more of the blend. So this run does both: one reinitialized layer, and the weight
refitted to match.

| setting | Run 9, current best | Run 11 |
|---|---|---|
| SVM | demojized, `--full-fit`, all 6,401 rows | same |
| MuRIL | demojized, `--folds 1`, 1 seed, 6 epochs, effective batch 16 | same |
| reinitialized layers | 2 | **1** |
| blend weight | 0.57 / 0.43, fitted in Run 8 | **refitted here** |
| decision threshold | 0.5, never tuned | **fitted here** |

## Why this needs a fold pass, and why it is worth it anyway

A blend weight cannot be fitted on a full-data run: there is nothing held out to fit it
against except the hidden labels. So stage 1 runs five folds for both components, stage 2
fits the weight and threshold on out-of-fold probabilities with a nested check, and only
then does stage 3 refit both components on all 6,401 rows and apply those values.

That costs about **3.4 hours**, not the eight an earlier draft would have, because folds
are run only for the configuration being shipped rather than for a two-way comparison.

Three things come out of it besides the submission:

* **MuRIL's own out-of-fold score**, which says whether one layer beats the `0.8073` TF-IDF
  floor on identical data
* **a fitted threshold**, measured on the floor as worth `+0.0035` for no GPU
* **`oof_probs.npy` for both components**, which the repository has never had for Task A.
  With them, every future blend weight, threshold or decode rule is seconds of CPU instead
  of hours of GPU

## Runtime

About **3.4 hours**: minutes for the SVM fold pass, ~160 for MuRIL's, then ~35 for the two
full-data refits. The nested search costs nothing.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Five-fold out-of-fold probabilities for both components

Both on split seed 42 so the matrices line up row for row; without that they cannot be
blended honestly. `--select last` throughout, because `best` picks each fold's checkpoint
using the rows it then reports.

In [ ]:
SVM_OOF, MURIL_OOF = "task_a_r11_svm_oof", "task_a_r11_muril_oof"

run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_OOF, "--demojize"], log=f"artifacts/logs/{SVM_OOF}.log")

run([sys.executable, "-u", "-m", "hastika.models.muril",
     "--tag", MURIL_OOF, "--model", "google/muril-base-cased",
     "--folds", "5", "--seeds", "42", "--epochs", "6",
     "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
     "--select", "last", "--reinit-layers", "1"],
    log=f"artifacts/logs/{MURIL_OOF}.log")

from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from hastika.common.preprocessing import clean

df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values
svm_oof = np.load(pathlib.Path("artifacts/runs") / SVM_OOF / "oof_probs.npy")
muril_oof = np.load(pathlib.Path("artifacts/runs") / MURIL_OOF / "oof_probs.npy")
assert len(svm_oof) == len(muril_oof) == len(y)

print(f"  SVM               OOF macro-F1 {f1_score(y, svm_oof.argmax(1), average='macro'):.4f}")
print(f"  MuRIL, one layer  OOF macro-F1 {f1_score(y, muril_oof.argmax(1), average='macro'):.4f}")
print("  TF-IDF floor reference: 0.8073 on these same folds")

## 2. Fit the weight and the threshold, nested

`w` is the SVM's share. Both `w` and the threshold are chosen on an inner split of each
fold's training rows and applied to rows that never influenced them, so the reported number
does not flatter itself. The in-sample sweep is printed for shape only.

If one layer has made MuRIL stronger, the fitted `w` should come out below Run 8's 0.57.

In [ ]:
GRID_W = np.arange(0.0, 1.001, 0.05)
GRID_T = np.arange(0.30, 0.71, 0.02)

def sc(w, t, idx):
    p = w * svm_oof[idx, 1] + (1 - w) * muril_oof[idx, 1]
    return f1_score(y[idx], (p > t).astype(int), average="macro")

allidx = np.arange(len(y))
print("in-sample, at threshold 0.50:")
for w in [0.0, 0.25, 0.43, 0.57, 0.75, 1.0]:
    print(f"  w_svm={w:.2f}  {sc(w, 0.50, allidx):.4f}")
bw, bt = max(((w, t) for w in GRID_W for t in GRID_T), key=lambda p: sc(p[0], p[1], allidx))
print(f"  best in-sample: w={bw:.2f} t={bt:.2f} -> {sc(bw, bt, allidx):.4f}  (optimistic)")

pred = np.zeros(len(y), dtype=int); picks = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
    itr, iva = next(StratifiedKFold(4, shuffle=True, random_state=7).split(X[tr], y[tr]))
    inner = tr[iva]
    w, t = max(((w, t) for w in GRID_W for t in GRID_T), key=lambda p: sc(p[0], p[1], inner))
    picks.append((round(w, 2), round(t, 2)))
    p = w * svm_oof[va, 1] + (1 - w) * muril_oof[va, 1]
    pred[va] = (p > t).astype(int)
nested = f1_score(y, pred, average="macro")

W_SVM = float(np.mean([w for w, _ in picks]))
THRESH = float(np.mean([t for _, t in picks]))
print(f"\nnested blend macro-F1 {nested:.4f}   per-fold picks {picks}")
print(f"  Run 8's fixed 0.57 at threshold 0.50, in-sample: {sc(0.57, 0.50, allidx):.4f}")
print(f"\nvalues for the final fit: SVM {W_SVM:.2f} / MuRIL {1 - W_SVM:.2f}, "
      f"threshold {THRESH:.2f}")
print("  Run 8 fitted 0.57 against a TWO-layer MuRIL; this fits a one-layer one")

## 3. Refit both components on all 6,401 rows

`--full-fit` and `--folds 1`. The weight and threshold from stage 2 are applied unchanged
and are **not** re-optimized here: the hidden validation labels must never touch a final
fit.

In [ ]:
SVM_FULL, MURIL_FULL = "task_a_r11_svm_full", "task_a_r11_muril_full"

run([sys.executable, "-u", "-m", "hastika.models.baseline_svm",
     "--task", "a", "--tag", SVM_FULL, "--demojize", "--full-fit"],
    log=f"artifacts/logs/{SVM_FULL}.log")
run([sys.executable, "-u", "-m", "hastika.models.muril",
     "--tag", MURIL_FULL, "--model", "google/muril-base-cased",
     "--folds", "1", "--seeds", "42", "--epochs", "6",
     "--bs", "8", "--grad-accum", "2", "--eval-bs", "32",
     "--select", "last", "--reinit-layers", "1"],
    log=f"artifacts/logs/{MURIL_FULL}.log")

import re
log = pathlib.Path(f"artifacts/logs/{MURIL_FULL}.log").read_text()
fits = re.findall(r"seed (\d+) FULL FIT, (\d+) rows", log)
print("full fits:", fits)
assert fits and all(int(n) == 6401 for _, n in fits), "a seed did not see all 6,401 rows"
assert "reinit=1" in log, "training log does not show one-layer reinitialization"

## 4. Build the submission, and check it without labels

Two label-free diagnostics, neither of which is a score. The class balance catches a
collapse, the only failure detectable without labels. The agreement says whether this is a
near-rerun of the current best or a genuinely different bet.

In [ ]:
svm_p = np.load(pathlib.Path("artifacts/runs") / SVM_FULL / "test_probs.npy")
muril_p = np.load(pathlib.Path("artifacts/runs") / MURIL_FULL / "test_probs.npy")
ids = pd.read_csv("data/raw/binary_validation_inputs.csv")
p = W_SVM * svm_p[:, 1] + (1 - W_SVM) * muril_p[:, 1]
labels = np.where(p > THRESH, "Hate", "Non-Hate")

out = pathlib.Path("artifacts/runs") / "task_a_r11_blend"
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"id": ids["id"], "label": labels}).to_csv(out / "predictions.csv", index=False)

train_rate = (train.iloc[keep]["Label"] == "Hate").mean()
print(f"predicted Hate rate {(labels == 'Hate').mean():.3f}   training prior {train_rate:.3f}")
prev = pathlib.Path("submissions/task_a/task_a_predictions.zip")
if prev.exists():
    with zipfile.ZipFile(prev) as z:
        old = pd.read_csv(z.open([n for n in z.namelist() if n.endswith(".csv")][0]))
    m = pd.DataFrame({"id": ids["id"], "label": labels}).merge(old, on="id",
                                                              suffixes=("_new", "_old"))
    print(f"agreement with the preserved Task A submission: "
          f"{(m['label_new'] == m['label_old']).mean():.3f}")

ZIP = "/kaggle/working/task_a_reinit1_refit_blend.zip"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "a", "--pred", str(out / "predictions.csv"), "--out", ZIP])
with zipfile.ZipFile(ZIP) as f:
    assert f.namelist() == ["predictions.csv"], f.namelist()
print("ready to upload:", ZIP)

## 5. Preserve outputs

`oof_probs.npy` for both components is the durable product. Task A has never had them.
Download them even if the submission loses: they make every future weight, threshold or
decode idea free.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_reinit1_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / pathlib.Path(ZIP).name)
shutil.copy2(out / "predictions.csv", OUT / "predictions.csv")
for tag in [SVM_OOF, MURIL_OOF, SVM_FULL, MURIL_FULL]:
    d = pathlib.Path("artifacts/runs") / tag
    for name in ["oof_probs.npy", "test_probs.npy"]:
        if (d / name).exists():
            shutil.copy2(d / name, OUT / f"{tag}_{name}")
    log = pathlib.Path(f"artifacts/logs/{tag}.log")
    if log.exists():
        shutil.copy2(log, OUT / log.name)
json.dump({"w_svm": W_SVM, "w_muril": 1 - W_SVM, "threshold": THRESH,
           "reinit_layers": 1, "seeds": ["42"], "nested_oof": nested, "picks": picks,
           "svm_oof": float(f1_score(y, svm_oof.argmax(1), average="macro")),
           "muril_oof": float(f1_score(y, muril_oof.argmax(1), average="macro"))},
          open(OUT / "config.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 6. After CodaBench scores it

Record in `docs/EXPERIMENTS.md` and `submissions/README.md`: the CodaBench score, the
refitted weight, the fitted threshold, and both components' out-of-fold scores. Preserve the
ZIP under `submissions/` the way the other entries are.

**If it beats `0.8187`, this recipe becomes the base** and every queued experiment stacks on
it: Run 12's TAPT, Run 13's external labels, more seeds, more epochs.

**If the gap is under about 3 points either way, it is not resolved** — one score on 806
rows carries roughly 1.5 points of standard deviation. Fall back on the nested out-of-fold
number from stage 2, which is measured on 6,401 rows and is the more reliable of the two.

Either way keep the stored `oof_probs.npy`. The next time a weight or threshold needs
deciding, it will not cost a GPU session.